# 03 — Dyna-Q Tuned (n=20, lr=0.2) Visualization

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 110

# ── Paths ─────────────────────────────────────────────────────────────────────
ROOT      = os.path.abspath('..')
DATA_DIR  = os.path.join(ROOT, 'data')
PLOTS_DIR = os.path.join(ROOT, 'plots')

# ── Config ────────────────────────────────────────────────────────────────────
ENVS = [
    ('empty_5x5',     'Empty 5×5'),
    ('empty_8x8',     'Empty 8×8'),
    ('empty_16x16',   'Empty 16×16'),
    ('doorkey_5x5',   'DoorKey 5×5'),
    ('doorkey_8x8',   'DoorKey 8×8'),
    ('doorkey_16x16', 'DoorKey 16×16'),
]
ENVS_EMPTY   = ENVS[:3]
ENVS_DOORKEY = ENVS[3:]

DECAYS = ['linear', 'exp', 'rbed_linear', 'rbed_exp']
DECAY_LABELS = {
    'linear':      'Linear Decay',
    'exp':         'Exponential Decay',
    'rbed_linear': 'RBED + Linear',
    'rbed_exp':    'RBED + Exponential',
}
DECAY_COLORS = {
    'linear':      '#2266CC',
    'exp':         '#22AA55',
    'rbed_linear': '#CC4422',
    'rbed_exp':    '#AA22AA',
}
DECAY_STYLES = {'linear':'-', 'exp':'--', 'rbed_linear':':', 'rbed_exp':'-.'}

ALGO_INFO = {
    'qlearning':   ('Q-Learning',           'royalblue',  '-'),
    'dynaq':       ('Dyna-Q (n=5)',          'green',      '--'),
    'dynaq_tuned': ('Dyna-Q Tuned (n=20)',  'teal',       '-.'),
    'a2c_td':      ('A2C (TD)',              'tomato',     ':'),
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def load(algo, env, decay):
    folder = os.path.join(DATA_DIR, algo)
    metrics = ['rewards','steps','epsilons','success','avg_q']
    paths = {m: os.path.join(folder, f'{algo}__{env}__{decay}__{m}.npy') for m in metrics}
    if not all(os.path.exists(p) for p in paths.values()): return None
    return {m: np.load(paths[m]) for m in metrics}

def smooth(x, w=100):
    if len(x) < w: return x
    return np.convolve(x, np.ones(w)/w, mode='valid')

def summary(st, last_n=500):
    if st is None: return None
    r, s, steps = st['rewards'], st['success'], st['steps']
    first = next((i for i,v in enumerate(s) if v > 0), None)
    peak  = float(np.max(np.convolve(s, np.ones(100)/100, 'valid'))) if len(s)>=100 else float(np.mean(s))
    return {
        'sr':        float(np.mean(s[-last_n:])),
        'sr100':     float(np.mean(s[-100:])),
        'avg_r':     float(np.mean(r[-last_n:])),
        'best_r':    float(np.max(r)),
        'avg_steps': float(np.mean(steps[-last_n:])),
        'first':     first,
        'peak_sr':   peak,
    }

def savefig(fig, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=130, bbox_inches='tight')
    print(f'Saved: {path}')

print("✅ Setup complete")

In [ ]:
ALGO  = 'dynaq_tuned'
LABEL = 'Dyna-Q Tuned (n=20, lr=0.2)'
COLOR = 'teal'
OUT   = os.path.join(PLOTS_DIR, 'dynaq_tuned')
os.makedirs(OUT, exist_ok=True)
ENV_COLORS = {
    'empty_5x5':'#1f77b4','empty_8x8':'#4aa3df','empty_16x16':'#aec7e8',
    'doorkey_5x5':'#d62728','doorkey_8x8':'#e07055','doorkey_16x16':'#f5b8b0',
}
LS_MAP = {'empty_5x5':'-','empty_8x8':'--','empty_16x16':':',
           'doorkey_5x5':'-','doorkey_8x8':'--','doorkey_16x16':':'}
print(f'Output: {OUT}')

## 1. Per Environment — 4 Decay Types

In [ ]:
for env_id, env_label in ENVS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{LABEL} — {env_label}', fontsize=13, fontweight='bold')
    ax_r, ax_sr, ax_eps = axes
    for decay in DECAYS:
        st = load(ALGO, env_id, decay)
        if st is None: continue
        c, ls, lbl = DECAY_COLORS[decay], DECAY_STYLES[decay], DECAY_LABELS[decay]
        sm = summary(st)
        ax_r.plot(smooth(st['rewards']),  color=c, ls=ls, lw=1.8, label=lbl)
        ax_sr.plot(smooth(st['success']), color=c, ls=ls, lw=1.8, label=lbl)
        ax_eps.plot(st['epsilons'],       color=c, ls=ls, lw=1.2, alpha=0.9, label=lbl)
        if sm and sm['first']: ax_sr.axvline(sm['first'], color=c, ls=':', lw=0.8, alpha=0.5)
    ax_r.axhline(0, color='k', ls='--', lw=0.6, alpha=0.4)
    for ax, ylabel, title in [(ax_r,'Total Reward (avg 100)','Training Curve'),
                               (ax_sr,'Success Rate (avg 100)','Success Rate'),
                               (ax_eps,'Epsilon ε','Exploration Rate')]:
        ax.set_xlabel('Episode'); ax.set_ylabel(ylabel)
        ax.set_title(title); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax_sr.set_ylim(-0.05, 1.05); ax_eps.set_ylim(0, 1.05)
    plt.tight_layout()
    savefig(fig, f'{OUT}/{ALGO}__{env_id}__decay_compare.png')
    plt.show()

## 2. Per Decay Type — 6 Environments

In [ ]:
for decay in DECAYS:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    fig.suptitle(f'{LABEL} — {DECAY_LABELS[decay]}', fontsize=13, fontweight='bold')
    ax_r, ax_sr, ax_st = axes
    for env_id, env_label in ENVS:
        st = load(ALGO, env_id, decay)
        if st is None: continue
        c, ls = ENV_COLORS[env_id], LS_MAP[env_id]
        ax_r.plot(smooth(st['rewards']),  color=c, ls=ls, lw=1.8, label=env_label)
        ax_sr.plot(smooth(st['success']), color=c, ls=ls, lw=1.8, label=env_label)
        ax_st.plot(smooth(st['steps']),   color=c, ls=ls, lw=1.4, label=env_label)
    ax_r.axhline(0, color='k', ls='--', lw=0.6, alpha=0.4)
    ax_r.set_title('Training Curve');  ax_r.set_ylabel('Total Reward (avg 100)')
    ax_sr.set_title('Success Rate');   ax_sr.set_ylabel('Success Rate (avg 100)'); ax_sr.set_ylim(-0.05,1.05)
    ax_st.set_title('Steps/Episode');  ax_st.set_ylabel('Steps (avg 100)')
    for ax in axes: ax.set_xlabel('Episode'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout()
    savefig(fig, f'{OUT}/{ALGO}__{decay}__env_compare.png')
    plt.show()

## 3. Summary Heatmap

In [ ]:
sr_mat = np.full((len(DECAYS), len(ENVS)), np.nan)
r_mat  = np.full((len(DECAYS), len(ENVS)), np.nan)
for i, decay in enumerate(DECAYS):
    for j, (env_id, _) in enumerate(ENVS):
        sm = summary(load(ALGO, env_id, decay))
        if sm: sr_mat[i,j]=sm['sr']; r_mat[i,j]=sm['avg_r']
fig, axes = plt.subplots(1, 2, figsize=(16, 4))
fig.suptitle(f'{LABEL} — Summary Heatmap', fontsize=13, fontweight='bold')
for ax, mat, title, fmt in [(axes[0],sr_mat,'Success Rate','sr'),(axes[1],r_mat,'Avg Reward','r')]:
    vmin=np.nanmin(mat); vmax=np.nanmax(mat)
    im = ax.imshow(mat, cmap='RdYlGn', aspect='auto', vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(ENVS))); ax.set_xticklabels([e[1] for e in ENVS], rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(DECAYS))); ax.set_yticklabels([DECAY_LABELS[d] for d in DECAYS], fontsize=9)
    ax.axvline(2.5, color='white', lw=2.5); ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(len(DECAYS)):
        for j in range(len(ENVS)):
            v=mat[i,j]
            if np.isnan(v): continue
            txt=f'{v:.0%}' if fmt=='sr' else f'{v:.2f}'
            nv=(v-vmin)/(vmax-vmin+1e-9)
            ax.text(j,i,txt,ha='center',va='center',fontsize=8,fontweight='bold',color='white' if nv<0.35 else 'black')
    plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
plt.tight_layout()
savefig(fig, f'{OUT}/{ALGO}__heatmap.png')
plt.show()

## 4. Summary Table

In [ ]:
print(f"{'Env':<16} {'Decay':<22} {'SR(500)':>8} {'SR(100)':>8} {'AvgR':>8} {'First':>8}")
print("-"*75)
for env_id, env_label in ENVS:
    for decay in DECAYS:
        sm = summary(load(ALGO, env_id, decay))
        if not sm: continue
        fe = f"ep{sm['first']}" if sm['first'] else 'Never'
        print(f"  {env_label:<14} {DECAY_LABELS[decay]:<22} {sm['sr']:>7.1%} {sm['sr100']:>7.1%} {sm['avg_r']:>8.3f} {fe:>8}")
    print()